In [ ]:
# April 15, 2026
# working to verify the dim, tau scaling for HMC
# import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
from jax import jit, vmap

import jax.random as jr
import jax.numpy as jnp
from datatypes import QP, IntegratorConfig
from target import gen_perturb_precision
from hamiltonian import gaussian_hamiltonian
from sampler import hmc_sampler, chmc_sampler, extract_positions, extract_energy
from metrics import compute_accept_rate

import time

import numpy as np

In [5]:
# ========================================
# CHMC v HMC: fixed κ, τ sweep (optimized)
# ========================================
# initial key
key = jax.random.PRNGKey(1)

numdims = 8
dims = np.logspace(1, 8, numdims, endpoint=True, base=2, dtype=int)
c_step = 11
runs = 20
τcounts = np.array([5, 10, 20, 40, 80, 160])
τs = 1 / τcounts
numτs = len(τs)
perturb_val = jnp.arange(c_step) / ((c_step - 1) * 2)

mcmc_min_exp = 2
mcmc_max_exp = 3
mcmc_iter_count = 7
lens = jnp.logspace(mcmc_min_exp, mcmc_max_exp, mcmc_iter_count, base=10, dtype=int)

T = 1.0
tol = 1e-2
max_iter = 2

key1, key2, key3 = jr.split(key, 3)
keyring     = jr.split(key1, numτs)
chmckeyring = jr.split(key2, numτs * len(lens) * runs)
hmckeyring  = jr.split(key3, numτs * len(lens) * runs)

hmc_chainring  = []
chmc_chainring = []
true_matrices  = []

totaltime = time.time()

jhmc_sampler  = jit(hmc_sampler,  static_argnums=(2, 3))
jchmc_sampler = jit(chmc_sampler, static_argnums=(2, 3, 4))

def run_single_hmc(carry, run_key):
    """
    Scan body: runs one HMC chain of length mainnum_samples.
    carry  = (init_sample, H_flat, config, mainnum_samples)
    run_key: a single PRNG key that we split into per-sample keys
    """
    init_sample, H_flat, config, mainnum_samples = carry
    sample_keys = jr.split(run_key, mainnum_samples)
    return carry, jhmc_sampler(init_sample, sample_keys, H_flat, config)

for l, dim in enumerate(dims):
    print(f"dim {l+1}/{len(dims)}  (d={dim})")

    Mass_inv   = jnp.eye(dim)
    target_mat = jnp.eye(dim)
    true_matrices.append(target_mat)

    H      = gaussian_hamiltonian(target_mat, mass_inv=Mass_inv)
    H_flat = lambda qp_flat, H=H: H(QP.from_array(qp_flat))

    for k, tau_val in enumerate(τs):
        N      = int(jnp.ceil(T / tau_val))
        config = IntegratorConfig(τ=tau_val, T=T, N=N, tol=tol, max_iter=max_iter)

        qp_init     = jr.normal(keyring[k], shape=(2 * dim,))
        init_sample = [qp_init, 1, False]

        for j, mainnum_samples in enumerate(lens):
            mainnum_samples = int(mainnum_samples)

            flat_base = k * (len(lens) * runs) + j * runs
            # shape (runs, 2) — one root key per run, no aliasing
            run_keys = hmckeyring[flat_base : flat_base + runs]

            # vmap over runs: each run gets its own root key,
            # splits it internally into mainnum_samples per-step keys
            batched_hmc = jit(
                vmap(
                    lambda rk: jhmc_sampler(
                        init_sample,
                        jr.split(rk, mainnum_samples),
                        H_flat,
                        config,
                    )
                )
            )
            samples = batched_hmc(run_keys)        # shape (runs, ...)
            jax.block_until_ready(samples)

            # extract positions for each run and store
            for i in range(runs):
                run_sample = jax.tree.map(lambda x: x[i], samples)
                hmc_chainring.append(extract_positions(run_sample, accepted_only=True))

    print(f"  τ sweep done — cumulative: {time.time() - totaltime:.3f}s")

dim 1/8  (d=2)
  τ sweep done — cumulative: 19.884s
dim 2/8  (d=4)
  τ sweep done — cumulative: 45.072s
dim 3/8  (d=8)
  τ sweep done — cumulative: 76.776s
dim 4/8  (d=16)
  τ sweep done — cumulative: 110.537s
dim 5/8  (d=32)
  τ sweep done — cumulative: 159.525s
dim 6/8  (d=64)
  τ sweep done — cumulative: 256.909s
dim 7/8  (d=128)
  τ sweep done — cumulative: 536.484s
dim 8/8  (d=256)
  τ sweep done — cumulative: 1208.163s


In [6]:
# ========================================
# τChain Ring: Analysis
# (dim, numτs, len, run)
# ========================================
import metrics

hmc_cov_matrices = []


for chain in hmc_chainring:
    hmc_cov_matrices.append(metrics.cov(chain))

true_cov_matrices = [jnp.linalg.inv(m) for m in true_matrices]


In [ ]:
# ========================================
# τCHMC Chain Ring: Analysis
# Downstream analytics
# ring.shape: (dims, numτs, len, run)
# ========================================

hmc_cov_metric = []
counter = numτs*len(lens)*runs
for j, true_mat in enumerate(true_cov_matrices):
    for i in range(counter):
        hmc_cov_metric.append(metrics.maxtracediff(hmc_cov_matrices[j*counter + i], true_mat))
hmc_cov_metric = jnp.array(hmc_cov_metric)
hmc_cov_metric = hmc_cov_metric.reshape(numdims, numτs, len(lens), runs)
